# Notebook 11: Ablation Studies

**R5 — Evaluation, Ablations and Drift Correction (Technical Validation)**

This notebook presents the systematic ablation study for the Drift-Sense VIO project.
All evaluations use seeds **42, 43, 44** across **1,312 windows** (MH_01_easy: 720, MH_02_easy: 592)
under leave-one-sequence-out cross-validation.

In [ ]:
import sys
from pathlib import Path
root = Path('.').resolve().parent
sys.path.insert(0, str(root / 'src'))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

RESULTS = root / 'results' / 'ablations'
FIGURES = root / 'figures' / 'dl'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Working root:', root)
pd.set_option('display.float_format', '{:.4f}'.format)


## Section 1: Phase 1 Comparability Check

We first verify that the new harness reproduces the Phase 1 benchmark numbers bit-for-bit:
- **Mean baseline:** 4.53 cm
- **Tuned Lasso (E0):** 2.50 cm MAE (0.02504 m), R² = 0.76
- **Calibrated Physics prior:** R² = 0.82

Enforced by unit test `tests/test_dl_eval.py`.

In [ ]:
from vio_drift.evaluation.dl_eval import reproduce_phase1_benchmarks
benchmarks = reproduce_phase1_benchmarks()
print('=== Phase 1 Benchmark Reproduction ===')
for _, row in benchmarks[benchmarks['fold_id'] == 'mean'].iterrows():
    print(f"  {row['model']:15s}: MAE = {row['mae']*100:.2f} cm, R² = {row['r2']:.4f}")


## Section 2: Core Experiments E0–E7

Comparison of all model families against the Phase 1 Lasso baseline over seeds 42, 43, 44.
- **E0**: Phase 1 Lasso baseline (104 hand-crafted features)
- **E1**: 1D-CNN on IMU (standardized target: 2.60 cm)
- **E2**: LSTM on IMU (standardized target: 3.69 cm)
- **E3**: Hybrid H1 (IMU + features + physics: 2.64 cm with warm-up)
- **E4**: Hybrid H2 (Physics prior + learned residual: **2.15 cm**, beats Lasso!)
- **E4cnn**: Hybrid H2 with CNN encoder (**2.13 cm**)
- **E5**: Camera CNN (4.42 cm)
- **E6**: Hybrid H3 (IMU + Camera: 2.85 cm with warm-up)
- **E7**: Best Overall Hybrid (2.18 cm)

In [ ]:
df_e = pd.read_csv(RESULTS / 'experiments_e0_e7.csv')
display_cols = ['experiment_id', 'name', 'family', 'mae_mean_cm', 'mae_std_cm', 'rmse_mean_cm', 'r2_mean']
print(df_e[display_cols].to_string(index=False))


In [ ]:
lasso_mae = float(df_e[df_e['experiment_id']=='E0']['mae_mean_cm'].iloc[0])
COL = {
    'Phase 1 (ML / Physics)': '#2196F3',
    'DL (IMU-only)': '#FF9800',
    'Hybrid': '#4CAF50',
    'DL (Visual-only)': '#9C27B0',
}
fig, ax = plt.subplots(figsize=(12, 5))
colors = [COL.get(f, '#9E9E9E') for f in df_e['family']]
bars = ax.bar(df_e['experiment_id'], df_e['mae_mean_cm'],
              yerr=df_e['mae_std_cm'], capsize=4,
              color=colors, edgecolor='k', linewidth=0.6, zorder=3)
ax.axhline(lasso_mae, color='#E53935', linestyle='--', linewidth=1.4,
           label=f'Phase 1 Lasso baseline {lasso_mae:.2f} cm')
for bar, mae, std in zip(bars, df_e['mae_mean_cm'], df_e['mae_std_cm']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + std + 0.1,
            f'{mae:.2f}', ha='center', va='bottom', fontsize=8)
patches = [mpatches.Patch(color=v, label=k) for k, v in COL.items()]
ax.legend(handles=patches + [plt.Line2D([0], [0], color='#E53935', linestyle='--', label=f'Lasso {lasso_mae:.2f} cm')], fontsize=8)
ax.set_xlabel('Experiment'); ax.set_ylabel('MAE (cm)')
ax.set_title('E0–E7: Experiment Comparison (MAE ± std across seeds 42, 43, 44)')
ax.set_ylim(0, df_e['mae_mean_cm'].max() * 1.25)
ax.grid(axis='y', alpha=0.4, zorder=0)
plt.tight_layout(); plt.show()
print(f"\nKey finding: Hybrid H2 (E4) achieves {df_e[df_e.experiment_id=='E4']['mae_mean_cm'].values[0]:.2f} cm, outperforming Lasso ({lasso_mae:.2f} cm).")


## Section 3: A1 — Regularisation Ablation

Factorial ablation on Dropout (0.0 vs 0.3) and Weight Decay (0.0 vs 1e-4) on CNN1D (E1).

In [ ]:
df_a1 = pd.read_csv(RESULTS / 'regularization_a1.csv')
print(df_a1.to_string(index=False))


## Section 4: A2 — Data Scaling

MAE and R² as a function of chronological training data prefix (50%, 75%, 100%).

In [ ]:
df_a2 = pd.read_csv(RESULTS / 'data_scaling_a2.csv')
print(df_a2.to_string(index=False))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.errorbar(df_a2['fraction']*100, df_a2['mae_mean_cm'], yerr=df_a2['mae_std_cm'],
             marker='o', color='#1976D2', linewidth=1.8, capsize=4)
ax1.axhline(lasso_mae, color='#E53935', linestyle='--', label=f'Lasso {lasso_mae:.2f}')
ax1.set_xlabel('Training data prefix (%)'); ax1.set_ylabel('MAE (cm)')
ax1.set_title('A2: MAE vs Training Size'); ax1.legend(fontsize=8); ax1.grid(alpha=0.4)
ax2.plot(df_a2['fraction']*100, df_a2['r2_mean'], marker='s', color='#388E3C', linewidth=1.8)
ax2.set_xlabel('Training data prefix (%)'); ax2.set_ylabel('R²')
ax2.set_title('A2: R² vs Training Size'); ax2.grid(alpha=0.4)
plt.tight_layout(); plt.show()


## Section 5: A3 — Component Contributions

Decomposing the incremental benefit of physics prior, hand-crafted features, and learned residual.

In [ ]:
df_a3 = pd.read_csv(RESULTS / 'components_a3.csv')
print(df_a3.to_string(index=False))


## Section 6: A4 — Seed Spread

Assessing optimization stability across seeds 42, 43, 44.
Notice how Hybrid H2 exhibits tight clustering (std = 0.01 cm), whereas unconstrained deep networks exhibit larger variance.

In [ ]:
df_a4 = pd.read_csv(RESULTS / 'seeds_a4.csv')
print(df_a4[['model_id', 'n_seeds', 'mae_mean_cm', 'mae_std_cm', 'mae_min_cm', 'mae_max_cm', 'r2_mean']].to_string(index=False))


## Section 7: A5 — Flight Segments (Take-off vs Steady Cruise)

Decomposition into first-20% (take-off acceleration) vs remaining-80% (steady cruise).

In [ ]:
df_a5 = pd.read_csv(RESULTS / 'segments_a5.csv')
print(df_a5.to_string(index=False))


## Summary

| Experiment | MAE (cm) | R² | Note |
|---|---|---|---|
| E0 Lasso | 2.50 | 0.76 | Phase 1 baseline |
| E1 1D-CNN | 2.60 ± 0.12 | 0.72 | Standardized target |
| E2 LSTM | 3.69 ± 0.06 | 0.31 | Standardized target |
| E3 Hybrid H1 | 2.64 ± 0.03 | 0.73 | Warm-up min10 |
| **E4 Hybrid H2 (LSTM)** | **2.15 ± 0.01** | **0.82** | **Beats Lasso ✓** |
| **E4cnn Hybrid H2 (CNN)** | **2.13 ± 0.02** | **0.82** | **Best DL model ✓** |
| E5 Camera | 4.42 ± 0.12 | 0.07 | Visual-only |
| E6 Hybrid H3 | 2.85 ± 0.05 | 0.64 | IMU + Camera (warm-up) |
| E7 Best Hybrid | 2.18 ± 0.06 | 0.81 | Validation selected |

**Conclusion:** Hybrid H2 (physics prior + learned residual) is the top-performing model family, outperforming Phase 1 Lasso by over 14% while exhibiting minimal variance across random seeds.